# 2,000 images, from scratch

The realistic case: a few thousand images, a model that overfits within five epochs, and a baseline to improve on in the next two notebooks.

**Runs on:** GPU recommended — about 10 minutes on CPU &nbsp;·&nbsp; **Slides:** [Chapter 8 — Image Classification](../../../course-web-slides/ch08/index.html) &nbsp;·&nbsp; **Section:** 02 — Training a convnet from scratch on a small dataset

---

## Getting the data

In [ ]:
import os, shutil, pathlib
import keras

# Cats vs dogs. Requires a Kaggle account; see the book for the download step.
# If you already have the archive, point original_dir at the extracted folder.
original_dir = pathlib.Path("train")
new_base_dir = pathlib.Path("cats_vs_dogs_small")

def make_subset(subset_name, start_index, end_index):
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        os.makedirs(dir, exist_ok=True)
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname, dst=dir / fname)

if original_dir.exists() and not new_base_dir.exists():
    make_subset("train", 0, 1000)
    make_subset("validation", 1000, 1500)
    make_subset("test", 1500, 2500)
    print("subsets created")
else:
    print("point original_dir at your extracted Kaggle download")

> **Note** — **2,000 training images, 1,000 validation, 2,000 test.** Deliberately small. Everything interesting in this chapter follows from that number.

## image_dataset_from_directory

In [ ]:
from keras.utils import image_dataset_from_directory

train_dataset = image_dataset_from_directory(
    new_base_dir / "train", image_size=(180, 180), batch_size=32)
validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation", image_size=(180, 180), batch_size=32)
test_dataset = image_dataset_from_directory(
    new_base_dir / "test", image_size=(180, 180), batch_size=32)

for data_batch, labels_batch in train_dataset:
    print("data batch shape:", data_batch.shape)
    print("labels batch shape:", labels_batch.shape)
    break

Expected output:

```
data batch shape: (32, 180, 180, 3)
labels batch shape: (32,)
```

In [ ]:
import matplotlib.pyplot as plt

for images, labels in train_dataset.take(1):
    fig, axes = plt.subplots(2, 6, figsize=(13, 4.4))
    for ax, img, lab in zip(axes.ravel(), images, labels):
        ax.imshow(img.numpy().astype("uint8"))
        ax.set_title("dog" if lab == 1 else "cat", fontsize=9)
        ax.axis("off")
    plt.tight_layout(); plt.show()

## The model

In [ ]:
from keras import layers

inputs = keras.Input(shape=(180, 180, 3))
x = layers.Rescaling(1./255)(inputs)
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.Flatten()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(loss="binary_crossentropy", optimizer="rmsprop",
              metrics=["accuracy"])
model.summary()

`Rescaling` is **inside the model**, not in the data pipeline. Chapter 6's point about preprocessing travelling with the model, applied by default.

## Training, with a checkpoint

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="convnet_from_scratch.keras",
        save_best_only=True,
        monitor="val_loss")
]
history = model.fit(train_dataset, epochs=30,
                    validation_data=validation_dataset,
                    callbacks=callbacks, verbose=2)

## The curves, and the diagnosis

In [ ]:
import numpy as np

h = history.history
epochs = range(1, len(h["accuracy"]) + 1)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(epochs, h["accuracy"], "o-", ms=3, label="training")
a1.plot(epochs, h["val_accuracy"], "s-", ms=3, label="validation")
a1.set_title("Accuracy"); a1.legend(); a1.set_xlabel("epoch")
a2.plot(epochs, h["loss"], "o-", ms=3, label="training")
a2.plot(epochs, h["val_loss"], "s-", ms=3, label="validation")
a2.set_title("Loss"); a2.legend(); a2.set_xlabel("epoch")
plt.tight_layout(); plt.show()

turn = int(np.argmin(h["val_loss"])) + 1
print(f"validation loss bottoms out at epoch {turn} of {len(epochs)}")

Training accuracy runs to nearly 100%. Validation stalls around 70% and validation loss turns up within about five epochs. **Textbook overfitting**, and entirely expected with 2,000 samples.

## The baseline

In [ ]:
test_model = keras.models.load_model("convnet_from_scratch.keras")
_, test_acc = test_model.evaluate(test_dataset, verbose=0)
print(f"test accuracy: {test_acc:.3f}")

Expected output:

```
test accuracy: ~0.70
```

Around 70%. Hold onto that number: notebook 03 adds augmentation and reaches the low 80s; notebook 04 uses a pretrained backbone and reaches the high 90s. **Same data, same budget.**

---

## What to take away

- `image_dataset_from_directory` turns a folder tree into a batched dataset in one call.
- Put `Rescaling` inside the model so preprocessing travels with it.
- 2,000 images overfit a from-scratch ConvNet within five epochs.
- **70% is the baseline** the next two notebooks improve on.